In [1]:
# Clone the repository
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding the cloud's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml

Cloning into 'arabic-itsm-classification'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 134 (delta 51), reused 119 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.59 MiB | 20.34 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/kaggle/working/arabic-itsm-classification
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.4 MB/

In [2]:
# Clear existing placeholder created by git
!rm -rf data/processed && mkdir -p data/processed

# Link using the EXACT path confirmed via ls -R
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/

!ls data/processed
# Expected output: label_encoders.pkl, test.csv, train.csv, val.csv

label_encoders.pkl  test.csv  train.csv  val.csv


In [3]:
!nvidia-smi

Thu Feb 26 03:03:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Full multi-task training on Kaggle T4 GPU
# Output: models/marbert_multi_task_best/  (5 heads: l1, l2, l3, priority, sentiment)
# This is the production model — single 621 MB forward pass for all 5 tasks.

!python scripts/train.py \
    --multi-task \
    --data-dir data/processed \
    --epochs 10 \
    --lr 1e-5 \
    --batch-size 16

Device: cuda | FP16: True
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Tasks: ['l1', 'l2', 'l3', 'priority', 'sentiment'] | Classes: {'l1': 6, 'l2': 16, 'l3': 48, 'priority': 5, 'sentiment': 4}
tokenizer_config.json: 100%|███████████████████| 439/439 [00:00<00:00, 2.00MB/s]
vocab.txt: 1.10MB [00:00, 25.4MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 465kB/s]
pytorch_model.bin: 100%|██████████████████████| 654M/654M [00:02<00:00, 295MB/s]
model.safetensors:   0%|                             | 0.00/654M [00:00<?, ?B/s]
Loading weights:   0%|                                  | 0/199 [00:00<?, ?it/s]
Loading weights:   

In [5]:
import os, shutil

# Verify checkpoint was saved correctly
checkpoint_dir = 'models/marbert_multi_task_best'
print('Checkpoint files:', os.listdir(checkpoint_dir))

# Verify heads.pt contains all 5 tasks (10 keys = 5 tasks x weight+bias)
import torch
heads = torch.load(f'{checkpoint_dir}/heads.pt', map_location='cpu', weights_only=False)
print('Head keys:', sorted(heads.keys()))
assert len(heads.keys()) == 10, f'Expected 10 keys (5 tasks x 2), got {len(heads.keys())}'
print('\nVerification passed: 5 joint heads found (l1, l2, l3, priority, sentiment)')

# Create a compressed archive for easy download
shutil.make_archive('/kaggle/working/marbert_multi_task_best', 'zip', 'models', 'marbert_multi_task_best')
print('\nDownload archive: /kaggle/working/marbert_multi_task_best.zip')
print('Place in: arabic-itsm-classification/models/marbert_multi_task_best/')
print('Also copy to: arabic-itsm-server/models/marbert_multi_task_best/ for serving')

Checkpoint files: ['config.json', 'model.safetensors', 'heads.pt', 'tokenizer.json', 'tokenizer_config.json']
Head keys: ['l1.1.bias', 'l1.1.weight', 'l2.1.bias', 'l2.1.weight', 'l3.1.bias', 'l3.1.weight', 'priority.1.bias', 'priority.1.weight', 'sentiment.1.bias', 'sentiment.1.weight']

Verification passed: 5 joint heads found (l1, l2, l3, priority, sentiment)

Download archive: /kaggle/working/marbert_multi_task_best.zip
Place in: arabic-itsm-classification/models/marbert_multi_task_best/
Also copy to: arabic-itsm-server/models/marbert_multi_task_best/ for serving
